In [ ]:
import os, sys, pickle, warnings
import numpy as np
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

base_path = os.path.sep.join(os.path.abspath("__file__").split(os.path.sep)[:-2])
sys.path.insert(0, os.path.join(base_path, "code/functions"))

from linear_shock_functions import data_path, pklf_name, load_PF_starts
from stat_functions import *

# Brian2's wildcard import brings in a couple of numpy-namespace helpers (e.g. `isin`)
# used below; the spiking-network simulation itself now lives in run_offline.py.
from brian2 import *

print("Imports OK")

Imports OK


## Parameters

In [18]:
# ── track layout (matches linear_shock_variables) ──────────────────────────
num_states      = 8
num_CA3_neurons = num_states * 400   # cell_per_unit = 400
state_position  = np.array([[0, s + 0.5] for s in range(num_states)])  # centres

# ── states (1-indexed as used in the description) ─────────────────────────
FATIGUE_STATE_1IDX = [1,2,3]   # states 1-3 neurons are fatigued
CUE_STATE_1IDX     = 4   # cue is injected at state 4 neurons
FATIGUE_STATE      = [s - 1 for s in FATIGUE_STATE_1IDX]   # 0-indexed
CUE_STATE          = CUE_STATE_1IDX     - 1   # 0-indexed = 3

# ── simulation duration (matches the offline replay run in generate_data.ipynb) ────
sim_duration  = 20000   # ms

# ── Wu et al. exposure directories ─────────────────────────────────────────
N_TRIALS   = 7
trial_dirs = [os.path.join(data_path, "trial%d" % i) for i in range(N_TRIALS)]

print("Fatigue state (0-indexed):", FATIGUE_STATE, "→ state centre", state_position[FATIGUE_STATE])
print("Cue    state  (0-indexed):", CUE_STATE,     "→ state centre", state_position[CUE_STATE])


Fatigue state (0-indexed): [0, 1, 2] → state centre [[0.  0.5]
 [0.  1.5]
 [0.  2.5]]
Cue    state  (0-indexed): 3 → state centre [0.  3.5]


## Helper functions

In [19]:
def assign_state(neuron_id, PF_dict, state_position):
    """Return the 0-indexed state whose centre is closest to neuron_id's PF."""
    if neuron_id not in PF_dict:
        return -1
    pos   = np.array(PF_dict[neuron_id])
    dists = np.linalg.norm(state_position - pos, axis=1)
    return int(np.argmin(dists))

print("Helper functions defined")

Helper functions defined


## Load simulation results for all 10 Wu et al. shock exposures

Generated by `generate_data.ipynb` (`run_offline.simulate_fatigued_replay`); see the note under "Parameters" above.


## Analysis: forward vs. backward spike rate

In [ ]:
from tqdm import tqdm

PF_dict = load_PF_starts()
results = []
for trial in tqdm(range(N_TRIALS),desc="Trial: "):
    exp_dir = trial_dirs[trial]

    temp_f = dict(np.load(os.path.join(exp_dir, "CA3_replay_wu_fatigue.npz")))
    results.append({
        "spike_times":     temp_f["spike_times_CA3_PC"],
        "spiking_neurons": temp_f["spiking_neurons_CA3_PC"],
        "rate":            temp_f["rate_CA3_PC"],
        "PF_dict":         PF_dict
    })


Trial: 100%|██████████| 7/7 [00:00<00:00, 420.47it/s]


In [25]:
# ── Figure: 3-panel comparison ────────────────────────────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(5, 3))

x_exp   = np.arange(N_TRIALS)
width   = 0.35
COLORS  = {"forward": "steelblue", "backward": "tomato", "unclear": "lightgrey"}
COND_EDGE = ["black", "dimgrey"]

ahead_rate  = np.zeros(N_TRIALS); behind_rate = np.zeros(N_TRIALS)
# ── Panel 1: total count ────────────────────

ax = axs[0]
for trial_idx in range(N_TRIALS):
    r          = results[trial_idx]
    PF_dict    = r["PF_dict"]
    spike_nids = r["spiking_neurons"].astype(int)

    # Count spikes per state
    counts = np.zeros(num_states, dtype=int)
    for nid in spike_nids:
        s = assign_state(nid, PF_dict, state_position)
        if 0 <= s < num_states:
            counts[s] += 1

    ahead_rate[trial_idx]  = counts[CUE_STATE + 1:].sum()/(400*4 * sim_duration/1000)   # spikes per neuron per second
    behind_rate[trial_idx] = counts[:CUE_STATE].sum()/(400*3 * sim_duration/1000)    # spikes per neuron per second


jitter = 0.04 * (np.random.default_rng(7).random(N_TRIALS) - 0.5)
ax.scatter(np.zeros(N_TRIALS) + jitter, behind_rate, color="tomato",      s=40, zorder=3)
ax.scatter(np.ones(N_TRIALS)  + jitter, ahead_rate, color="steelblue", s=40, zorder=3)
for i in range(N_TRIALS):
    ax.plot([jitter[i], 1 + jitter[i]], [behind_rate[i], ahead_rate[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [behind_rate.mean(), ahead_rate.mean()],
            yerr=[behind_rate.std() / np.sqrt(N_TRIALS),
                  ahead_rate.std()  / np.sqrt(N_TRIALS)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Behind", "Ahead"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Firing rate")
ax.set_title("SZ vs. non-SZ replay", fontsize=10)

# ── Panel 2: backward-state ("behind") firing rate, fatigued vs. control ────
ax = axs[1]

behind_rate_ctrl = np.full(N_TRIALS, np.nan)
for trial_idx in range(N_TRIALS):
    exp_dir   = trial_dirs[trial_idx]
    ctrl_path = os.path.join(exp_dir, "CA3_replay_wu_ctrl.npz")
    if not os.path.exists(ctrl_path):
        continue
    PF_dict    = results[trial_idx]["PF_dict"]
    ctrl_data  = np.load(ctrl_path, allow_pickle=True)
    spike_nids = ctrl_data["spiking_neurons_CA3_PC"].astype(int)

    counts = np.zeros(num_states, dtype=int)
    for nid in spike_nids:
        s = assign_state(nid, PF_dict, state_position)
        if 0 <= s < num_states:
            counts[s] += 1

    behind_rate_ctrl[trial_idx] = counts[:CUE_STATE].sum()/(400*3 * sim_duration/1000)   # spikes per neuron per second

ax.scatter(np.zeros(N_TRIALS) + jitter, behind_rate_ctrl, color="grey",   s=40, zorder=3)
ax.scatter(np.ones(N_TRIALS)  + jitter, behind_rate,      color="tomato", s=40, zorder=3)
for i in range(N_TRIALS):
    ax.plot([jitter[i], 1 + jitter[i]], [behind_rate_ctrl[i], behind_rate[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [np.nanmean(behind_rate_ctrl), behind_rate.mean()],
            yerr=[np.nanstd(behind_rate_ctrl) / np.sqrt(N_TRIALS),
                  behind_rate.std()           / np.sqrt(N_TRIALS)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Firing rate")
ax.set_title("Behind-state replay: fatigued vs. control", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(data_path, "wu_FR_comparison.svg"),format='svg',
            bbox_inches="tight")
plt.show()

# ── Stats ─────────────────────────────────────────────────────────────────────

metrics = [
    ('Spike rate-behind vs. ahead',   behind_rate,  ahead_rate),
    ('Spike rate-without vs. with adaptation',   behind_rate_ctrl,  behind_rate),
]

for label, x, y in metrics:
    # drop NaN seeds for bias comparison
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f'{label:20s}: BW={xm.mean():.3f}±{xm.std(ddof=1):.3f}  '
          f'Shock={ym.mean():.3f}±{ym.std(ddof=1):.3f}  '
          f'\nt={t_stat:.3f}  p={p_val:.4f}  d_z={d_z:.3f}  '
          f'CI=[{ci_lo:.3f}, {ci_hi:.3f}]')


Spike rate-behind vs. ahead: BW=0.132±0.045  Shock=0.308±0.108  
t=-3.844  p=0.0085  d_z=-1.453  CI=[-0.289, -0.064]
Spike rate-without vs. with adaptation: BW=0.418±0.235  Shock=0.132±0.045  
t=3.742  p=0.0096  d_z=1.414  CI=[0.099, 0.474]


## Raster plots (one per exposure)

In [28]:
def neuron_to_state_col(nid, PF_dict):
    """Return the column (track position) of neuron `nid`'s place field."""
    return PF_dict.get(nid, [0, -1])[1]


fig2, axes2 = plt.subplots(2, 5, figsize=(22, 8))
axes2 = axes2.flatten()

for trial_idx in range(N_TRIALS):
    r          = results[trial_idx]
    PF_dict    = r["PF_dict"]
    spike_t    = r["spike_times"]      # ms
    spike_nids = r["spiking_neurons"].astype(int)

    # Map neuron → place-field column position for the y-axis
    pf_cols = np.array([neuron_to_state_col(n, PF_dict) for n in spike_nids])

    ax = axes2[trial_idx]
    ax.scatter(spike_t, pf_cols, s=0.5, c="k", rasterized=True)
    ax.axhline(state_position[FATIGUE_STATE[-1]][1], color="red",    linestyle="--",
               linewidth=1.0, label="fatigue centre")
    ax.axhline(state_position[CUE_STATE][1],     color="orange", linestyle="--",
               linewidth=1.0, label="cue centre")
    ax.set_xlim(0, sim_duration)
    ax.set_ylim(-0.5, num_states - 0.5)
    ax.set_yticks(np.arange(num_states) + 0.5)
    ax.set_yticklabels([f"S{i+1}" for i in range(num_states)], fontsize=7)
    ax.set_title(f"Exposure {trial_idx}")
    ax.set_xlabel("Time (ms)")
    if trial_idx == 0:
        ax.legend(fontsize=7, loc="upper right")

fig2.suptitle(
    "CA3 spike raster (y = PF column)  |  fatigue: state 3  |  cue: state 4",
    fontsize=13,
)
plt.show()
plt.tight_layout()

plt.savefig(os.path.join(data_path, "wu_offline_fatigued_rasters.pdf"),
            bbox_inches="tight")
print("Raster plots saved.")

Raster plots saved.


## Control simulation results — no fatigue

Same network and cue as above, but state-3 neurons keep standard adaptation
(`a_PC`, `b_PC`, zero initial `w`). Generated together with the fatigued run by
`generate_data.ipynb` (`run_offline.simulate_fatigued_replay`, `control_trials=[7]`)
and saved to `CA3_replay_wu_ctrl_cue4.npz` — only trial 7 has a cached control run.


## Juxtaposed raster: fatigued (top) vs. control (bottom)

Loaded from saved `.npz` files.
**x-axis** = time (ms); **y-axis** = neuron ID (S1 at bottom → S8 at top, 400 neurons per state band).
Colours: steelblue = forward states (S5–S8), tomato = backward states (S1–S2),
red = fatigue state (S3), orange = cue state (S4).

In [30]:
CELL_PER_UNIT = 400   # matches global_variables.cell_per_unit

state_y_bounds_wu = [(s * CELL_PER_UNIT, (s + 1) * CELL_PER_UNIT)
                     for s in range(num_states)]  # 8 states

def wu_state_from_nid(nid):
    return min(int(nid) // CELL_PER_UNIT, num_states - 1)

def wu_state_color(s):
    if isin(s, FATIGUE_STATE):                       return "red"
    if s == CUE_STATE:                           return "orange"
    if s > CUE_STATE:                            return "steelblue"   # forward
    return "tomato"                                                     # backward


COND_FILES_WU = [
    "CA3_replay_wu_fatigue.npz",
    "CA3_replay_wu_ctrl.npz",
]
ROW_TITLES_WU = ["Fatigued (S3)", "Control (no fatigue)"]

fig7, axes7 = plt.subplots(
    2, N_TRIALS,
    figsize=(2.6 * N_TRIALS, 8),
    sharey=True,
    gridspec_kw={"hspace": 0.06, "wspace": 0.05},
)

for trial_idx in range(N_TRIALS):
    exp_dir = trial_dirs[trial_idx]

    for row, fname in enumerate(COND_FILES_WU):
        ax = axes7[row, trial_idx]

        d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
        spike_t    = d["spike_times_CA3_PC"]           # ms
        spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
        states_v   = np.array([wu_state_from_nid(n) for n in spike_nids])

        for s in range(num_states):
            mask = states_v == s
            if mask.any():
                ax.scatter(
                    spike_t[mask], spike_nids[mask],
                    s=0.3, c=wu_state_color(s), rasterized=True,
                    zorder=3 if s in (FATIGUE_STATE, CUE_STATE) else
                           2 if s > CUE_STATE else 1,
                )

        # state boundary lines
        for y0, _ in state_y_bounds_wu:
            ax.axhline(y0, color="black", linewidth=0.4, linestyle="--", alpha=0.35)

        ax.set_xlim(0, sim_duration)
        ax.set_ylim(0, num_CA3_neurons)
        ax.tick_params(axis="x", labelsize=6)

        if row == 0:
            ax.set_title(f"E{trial_idx}", fontsize=9, pad=2)
            ax.set_xticklabels([])
        else:
            ax.set_xlabel("Time (ms)", fontsize=7)

        if trial_idx == 0:
            ax.set_ylabel(ROW_TITLES_WU[row] + "\nneuron ID", fontsize=8)
        else:
            ax.tick_params(labelleft=False)

# ── ytick labels — set once after loop so sharey doesn't wipe them ────────────
ytick_pos_wu = [(y0 + y1) / 2 for y0, y1 in state_y_bounds_wu]
for row in range(2):
    ax0 = axes7[row, 0]
    ax0.set_yticks(ytick_pos_wu)
    ax0.set_yticklabels([f"S{s+1}" for s in range(num_states)], fontsize=6)
    for tick, s in zip(ax0.get_yticklabels(), range(num_states)):
        tick.set_color(wu_state_color(s))

from matplotlib.patches import Patch as _Patch
leg7 = [
    _Patch(facecolor="tomato",     label="Backward S1-S2"),
    _Patch(facecolor="red",        label="Fatigue  S3"),
    _Patch(facecolor="orange",     label="Cue      S4"),
    _Patch(facecolor="steelblue",  label="Forward  S5-S8"),
]
axes7[0, -1].legend(handles=leg7, fontsize=7, loc="upper right",
                    bbox_to_anchor=(1.0, 1.0), framealpha=0.7)

fig7.suptitle(
    "Wu et al. CA3 raster  |  y = neuron ID (S1 bottom → S8 top)  |  "
    "top: fatigued (S3)  |  bottom: control",
    fontsize=11, y=1.01,
)
plt.savefig(os.path.join(data_path, "wu_juxtaposed_rasters.pdf"),
            bbox_inches="tight")
plt.show()
print("Saved → wu_juxtaposed_rasters.pdf")

Saved → wu_juxtaposed_rasters.pdf


In [ ]:
targ_trial = 2

CELL_PER_UNIT = 400   # matches global_variables.cell_per_unit

state_y_bounds_wu = [(s * CELL_PER_UNIT, (s + 1) * CELL_PER_UNIT)
                     for s in range(num_states)]  # 8 states

fig7_2, axes7_2 = plt.subplots(
    2, 1,
    figsize=(12, 3),
    sharey=True,
    gridspec_kw={"hspace": 0.06, "wspace": 0.05},
)

trial_idx = targ_trial; exp_dir = trial_dirs[trial_idx]

for row, fname in enumerate(COND_FILES_WU):
    ax = axes7_2[row]

    d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
    spike_t    = d["spike_times_CA3_PC"]           # ms
    spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
    states_v   = np.array([wu_state_from_nid(n) for n in spike_nids])

    for s in range(num_states):
        mask = states_v == s
        if mask.any():
            ax.scatter(
                spike_t[mask], spike_nids[mask],
                s=0.3, c=wu_state_color(s), rasterized=True, marker='.',
                zorder=3 if s in (FATIGUE_STATE, CUE_STATE) else
                        2 if s > CUE_STATE else 1,
            )

    # state boundary lines
    for y0, _ in state_y_bounds_wu:
        ax.axhline(y0, color="black", linewidth=0.4, linestyle="--", alpha=0.35)

    ax.set_xlim(0, sim_duration)
    ax.set_ylim(0, num_CA3_neurons)
    ax.tick_params(axis="x", labelsize=6)

    if row == 0:
        ax.set_title(f"E{trial_idx}", fontsize=9, pad=2)
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Time (ms)", fontsize=7)

    if trial_idx == 0:
        ax.set_ylabel(ROW_TITLES_WU[row] + "\nneuron ID", fontsize=8)
    else:
        ax.tick_params(labelleft=False)

# ── ytick labels — set once after loop so sharey doesn't wipe them ────────────
ytick_pos_wu = [(y0 + y1) / 2 for y0, y1 in state_y_bounds_wu]
for row in range(2):
    ax0 = axes7_2[row]
    ax0.set_yticks(ytick_pos_wu)
    ax0.set_yticklabels([f"S{s+1}" for s in range(num_states)], fontsize=6)
    for tick, s in zip(ax0.get_yticklabels(), range(num_states)):
        tick.set_color(wu_state_color(s))

plt.show()
plt.savefig(os.path.join(data_path, "wu_juxtaposed_rasters_trial%d.pdf" % targ_trial),
            bbox_inches="tight")

print("Saved → wu_juxtaposed_rasters_trial%d.pdf" % targ_trial)

Saved → wu_juxtaposed_rasters_trial2.pdf


## Replay direction analysis: forward (→ S8) vs. backward (→ S1)

**Method**
1. Smooth the population rate with a Gaussian kernel (σ = 20 ms).
2. Detect burst intervals where the smoothed rate exceeds mean + 2 × SD,
   subject to a minimum duration (50 ms) and a minimum gap between bursts (100 ms).
3. For each burst, compute the **Pearson correlation** between spike time and
   place-field column position.  r > 0.3 → forward replay; r < −0.3 → backward.
4. Repeat for both the fatigued and control saved `.npz` files, then compare.

In [34]:
from scipy.ndimage import gaussian_filter1d
from scipy import stats

# ── replay detection helpers ──────────────────────────────────────────────────

def smooth_rate(rate_arr, dt_ms, sigma_ms=20.0):
    return gaussian_filter1d(rate_arr, sigma=sigma_ms / dt_ms)


def detect_burst_intervals(rate_smooth, times_ms,
                           threshold, min_dur_ms=50, min_gap_ms=100):
    """Return list of (t_start_ms, t_end_ms) for contiguous bursts above threshold."""
    above = rate_smooth > threshold
    if not above.any():
        return []
    changes = np.diff(above.astype(int))
    on_idx  = np.where(changes ==  1)[0] + 1
    off_idx = np.where(changes == -1)[0] + 1
    if above[0]:  on_idx  = np.concatenate([[0],            on_idx])
    if above[-1]: off_idx = np.concatenate([off_idx, [len(times_ms) - 1]])

    intervals = []
    for si, ei in zip(on_idx, off_idx):
        t0, t1 = times_ms[si], times_ms[ei]
        if t1 - t0 < min_dur_ms:
            continue
        if intervals and t0 - intervals[-1][1] < min_gap_ms:
            intervals[-1] = (intervals[-1][0], t1)   # merge
        else:
            intervals.append((t0, t1))
    return intervals


def classify_direction(burst_spike_t, burst_nids, PF_dict,
                       r_thresh=0.3, min_spikes=5):
    """
    Pearson r between spike time and PF column.
    Returns 'forward', 'backward', or 'unclear'.
    """
    pf_cols = np.array([PF_dict.get(int(n), [0, -1])[1] for n in burst_nids])
    valid   = pf_cols >= 0
    if valid.sum() < min_spikes:
        return "unclear"
    r = np.corrcoef(burst_spike_t[valid], pf_cols[valid])[0, 1]
    if np.isnan(r):  return "unclear"
    if r >  r_thresh: return "forward"
    if r < -r_thresh: return "backward"
    return "unclear"


def count_replay_directions(spike_t_ms, spike_nids, rate_arr, times_ms, PF_dict,
                            sigma_ms=20, thresh_factor=1.0,
                            min_dur_ms=100, min_gap_ms=100,
                            r_thresh=0.2, min_spikes=5):
    """
    Detect bursts in rate_arr, classify each burst's replay direction.
    Returns dict with counts for 'forward', 'backward', 'unclear'.
    """
    dt = float(times_ms[1] - times_ms[0]) if len(times_ms) > 1 else 1.0
    rate_sm   = smooth_rate(rate_arr, dt, sigma_ms)
    threshold = rate_sm.mean() + thresh_factor * rate_sm.std()
    bursts    = detect_burst_intervals(rate_sm, times_ms, threshold,
                                       min_dur_ms, min_gap_ms)
    counts = {"forward": 0, "backward": 0, "unclear": 0}
    times = {"forward": [], "backward": [], "unclear": []}
    for t0, t1 in bursts:
        mask = (spike_t_ms >= t0) & (spike_t_ms <= t1)
        if mask.sum() < min_spikes:
            continue
        direction = classify_direction(spike_t_ms[mask], spike_nids[mask],
                                       PF_dict, r_thresh, min_spikes)
        counts[direction] += 1
        times[direction].append((t0, t1))
    counts["n_bursts"] = len(bursts)
    return counts, times


# ── loop over both conditions ─────────────────────────────────────────────────

COND_LABELS = ["Fatigued", "Control"]

# shape: (2 conditions, N_TRIALS, 3 directions)
direction_labels = ["forward", "backward", "unclear"]
replay_dir_counts = np.zeros((2, N_TRIALS, 3), dtype=int)

file_dir = os.path.join("/data/suheecho/BTSP","data","linear_shock")
for cond_idx, (cond_label, fname) in enumerate(zip(COND_LABELS, COND_FILES_WU)):
    print(f"\n=== {cond_label} ===")
    for trial_idx in range(N_TRIALS):
        exp_dir = trial_dirs[trial_idx]

        d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
        spike_t    = d["spike_times_CA3_PC"]           # ms
        spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
        rate_arr   = d["rate_CA3_PC"]                  # Hz, shape (T,)

        # reconstruct time axis from sim_duration and rate length
        times_ms = np.linspace(0, sim_duration, len(rate_arr))

        counts, times = count_replay_directions(spike_t, spike_nids,
                                         rate_arr, times_ms, PF_dict)
        for di, dlabel in enumerate(direction_labels):
            replay_dir_counts[cond_idx, trial_idx, di] = counts[dlabel]

        print(f"  Exposure {trial_idx}: bursts={counts['n_bursts']}  "
              f"fwd={counts['forward']}  bwd={counts['backward']}  "
              f"unclear={counts['unclear']}")
        foldername = "trial%d"%trial_idx
        save_path = os.path.join(file_dir,foldername)
        np.savez_compressed(os.path.join(save_path, f"CA3_replay_{COND_LABELS[cond_idx]}_type_{trial_idx}.npz"), times=times, counts=counts)

print("\nDone.")
print("\nForward counts — fatigued:", replay_dir_counts[0, :, 0])
print("Forward counts — control: ", replay_dir_counts[1, :, 0])
print("Backward counts — fatigued:", replay_dir_counts[0, :, 1])
print("Backward counts — control: ", replay_dir_counts[1, :, 1])


=== Fatigued ===
  Exposure 0: bursts=5  fwd=5  bwd=0  unclear=0
  Exposure 1: bursts=4  fwd=4  bwd=0  unclear=0
  Exposure 2: bursts=6  fwd=4  bwd=1  unclear=1
  Exposure 3: bursts=3  fwd=2  bwd=1  unclear=0
  Exposure 4: bursts=7  fwd=7  bwd=0  unclear=0
  Exposure 5: bursts=6  fwd=6  bwd=0  unclear=0
  Exposure 6: bursts=5  fwd=4  bwd=1  unclear=0

=== Control ===
  Exposure 0: bursts=8  fwd=7  bwd=1  unclear=0
  Exposure 1: bursts=5  fwd=3  bwd=2  unclear=0
  Exposure 2: bursts=8  fwd=2  bwd=6  unclear=0
  Exposure 3: bursts=5  fwd=2  bwd=3  unclear=0
  Exposure 4: bursts=8  fwd=4  bwd=4  unclear=0
  Exposure 5: bursts=6  fwd=5  bwd=1  unclear=0
  Exposure 6: bursts=5  fwd=5  bwd=0  unclear=0

Done.

Forward counts — fatigued: [5 4 4 2 7 6 4]
Forward counts — control:  [7 3 2 2 4 5 5]
Backward counts — fatigued: [0 0 1 1 0 0 1]
Backward counts — control:  [1 2 6 3 4 1 0]


In [ ]:
# ── Figure: 3-panel comparison ────────────────────────────────────────────────
fig9, axes9 = plt.subplots(1, 2, figsize=(5, 3))

x_exp   = np.arange(N_TRIALS)
width   = 0.35
COLORS  = {"forward": "steelblue", "backward": "tomato", "unclear": "lightgrey"}
COND_EDGE = ["black", "dimgrey"]

# ── Panel 1: total count ────────────────────
ax = axes9[0]
total_fat  = np.maximum(replay_dir_counts[0, :, 0] + replay_dir_counts[0, :, 1], 1)
total_ctrl = np.maximum(replay_dir_counts[1, :, 0] + replay_dir_counts[1, :, 1], 1)


jitter = 0.04 * (np.random.default_rng(7).random(N_TRIALS) - 0.5)
ax.scatter(np.zeros(N_TRIALS) + jitter, total_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(N_TRIALS)  + jitter, total_fat,  color="tomato", s=40, zorder=3)
for i in range(N_TRIALS):
    ax.plot([jitter[i], 1 + jitter[i]], [total_ctrl[i], total_fat[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [total_ctrl.mean(), total_fat.mean()],
            yerr=[total_ctrl.std() / np.sqrt(N_TRIALS),
                  total_fat.std()  / np.sqrt(N_TRIALS)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 15); ax.set_yticks(np.arange(0, 16, 5))
ax.set_ylabel("Replay count")
ax.set_title("Total replay count (forward + backward)")

# ── Panel 2: forward bias = fwd/(fwd+bwd), paired scatter ────────────────────
ax = axes9[1]
bias_fat   = replay_dir_counts[0, :, 0] / total_fat
bias_ctrl  = replay_dir_counts[1, :, 0] / total_ctrl

jitter = 0.1 * (np.random.default_rng(7).random(N_TRIALS) - 0.5)
ax.scatter(np.zeros(N_TRIALS) + jitter, bias_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(N_TRIALS)  + jitter, bias_fat,  color="tomato", s=40, zorder=3)
for i in range(N_TRIALS):
    ax.plot([jitter[i], 1 + jitter[i]], [bias_ctrl[i], bias_fat[i]],
            color="lightgrey", linewidth=1, zorder=1)
ax.errorbar([0, 1],
            [bias_ctrl.mean(), bias_fat.mean()],
            yerr=[bias_ctrl.std() / np.sqrt(N_TRIALS),
                  bias_fat.std()  / np.sqrt(N_TRIALS)],
            fmt="D", color="black", markersize=10, capsize=5, zorder=5)
# ax.boxplot([bias_ctrl, bias_fat], positions=[0, 1], patch_artist=True)
# ax.axhline(0.5, color="black", linestyle=":", linewidth=0.8, label="50 % chance")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Fatigued"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.1, 1.1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("Forward / (Forward + Backward)")
ax.set_title("Forward bias (paired)")

plt.tight_layout()
plt.savefig(os.path.join(data_path, "wu_replay_direction_comparison.svg"),format='svg',
            bbox_inches="tight")
plt.show()

# ── Stats ─────────────────────────────────────────────────────────────────────

metrics = [
    ('Total count', replay_dir_counts[0].sum(-1), replay_dir_counts[1].sum(-1)),
    ('SZ count', replay_dir_counts[0, :, 0], replay_dir_counts[1, :, 0]),
    ('non-SZ count', replay_dir_counts[0, :, 1], replay_dir_counts[1, :, 1]),
]

for label, x, y in metrics:
    # drop NaN seeds for bias comparison
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f'{label}: Control={xm.mean():.3f}±{xm.std(ddof=1):.3f}  '
          f'Fatigued={ym.mean():.3f}±{ym.std(ddof=1):.3f}  '
          f'\nt={t_stat:.3f}  p={p_val:.4f}  d_z={d_z:.3f}  '
          f'CI=[{ci_lo:.3f}, {ci_hi:.3f}]')


Total count: Control=5.143±1.345  Fatigued=6.429±1.512  
t=-3.057  p=0.0223  d_z=-1.155  CI=[-2.315, -0.257]
SZ count: Control=4.571±1.618  Fatigued=4.000±1.826  
t=0.880  p=0.4128  d_z=0.333  CI=[-1.018, 2.161]
non-SZ count: Control=0.429±0.535  Fatigued=2.429±2.070  
t=-2.646  p=0.0382  d_z=-1.000  CI=[-3.850, -0.150]
